# SIA P1 — Device-Actions LoRA Training (Google Cloud)

Google Cloud version of the Colab notebook. It fine-tunes the SIA
device-actions LoRA adapter using the plain Hugging Face stack
(`transformers` + `peft` + `trl`) via `sia-lab/posttrain/train_gcp.py` —
no unsloth toolchain required.

**Where to run this:** a **Vertex AI Workbench** instance with a GPU
(L4 or T4), or any GCP GPU VM built from a CUDA Deep Learning image with
JupyterLab. Select a **GPU** kernel/runtime before running.

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Get the code + data
Clone the repo (use the branch that carries the GCP trainer).

In [ ]:
import os
os.chdir('/home/jupyter')  # Vertex Workbench home; use /content on Colab
!rm -rf SIA
!git clone --branch claude/fix-bugs-ci-status-136uzp https://github.com/skmandal3240/SIA.git
os.chdir('SIA')
print('cwd:', os.getcwd())

## Step 3 — Install the training deps
Deep Learning / Vertex images already ship a CUDA `torch`; the rest are light.

In [ ]:
!pip install -q -r sia-lab/posttrain/requirements-train.txt

## Step 4 — (Gated base models only) authenticate to Hugging Face
`meta-llama/*` needs a token + accepted license. Skip this cell if you use
a public mirror such as `unsloth/Llama-3.2-1B-Instruct`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 5 — Validate data + config (no GPU spent)

In [ ]:
!python3 sia-lab/posttrain/train_gcp.py --dry-run

## Step 6 — Train, evaluate, and export the adapter
Set `OUTPUT_DIR` to a `gs://` bucket path to persist the adapter beyond
this instance, or a local path to keep it on disk. `--merge` also writes a
merged fp16 model for serving.

In [ ]:
BASE = 'meta-llama/Llama-3.2-1B-Instruct'  # or 'unsloth/Llama-3.2-1B-Instruct'
OUTPUT_DIR = 'gs://YOUR_BUCKET/sia/device_actions_lora'  # or a local path

!python3 sia-lab/posttrain/train_gcp.py \
    --base $BASE \
    --train sia-lab/posttrain/data/device_actions_train.json \
    --val   sia-lab/posttrain/data/device_actions_val.json \
    --output-dir $OUTPUT_DIR \
    --epochs 3 --merge

## Step 7 — Inspect the held-out accuracy

In [ ]:
import json, glob
for p in glob.glob('sia-lab/posttrain/outputs/**/eval.json', recursive=True):
    print(p, json.load(open(p)))

## Step 8 — Download the adapter locally (optional)
If you trained to a local path (not `gs://`), zip and pull it down.

In [ ]:
import shutil, os
src = 'sia-lab/posttrain/outputs/device_actions_lora'
if os.path.isdir(src):
    shutil.make_archive('sia_device_actions_lora', 'zip', src)
    print('created sia_device_actions_lora.zip — download it from the file browser')
else:
    print('adapter went to a gs:// bucket; fetch it with: gsutil -m cp -r', os.environ.get('OUTPUT_DIR',''))